In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import *
from tensorflow.keras.optimizers import *
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

In [5]:
import os
import pandas as pd

# Define directories
IMAGE_DIR = 'dataset/images'
LABEL_DIR = 'dataset/labels'

# Function to load data
def load_data(image_dir, label_dir):
    paths = []
    labels = []
    
    label_files = [f for f in os.listdir(label_dir) if f.endswith('.txt')]
    
    for file in label_files:
        image_path = os.path.join(image_dir, file.replace('.txt', '.jpg'))
        label_path = os.path.join(label_dir, file)
        
        if os.path.exists(image_path):
            with open(label_path, 'r') as f:
                label_info = f.readline().strip().split()
                
                if len(label_info) < 5:
                    continue  # Skip this entry if it doesn't have enough elements
                
                class_index = int(label_info[0])
                bbox_x = float(label_info[1])
                bbox_y = float(label_info[2])
                bbox_width = float(label_info[3])
                bbox_height = float(label_info[4])
                
            paths.append(image_path)
            labels.append([class_index, bbox_x, bbox_y, bbox_width, bbox_height])
    
    return pd.DataFrame({'path': paths, 'label': labels})

# Load data
train_df = load_data(os.path.join(IMAGE_DIR, 'train'), os.path.join(LABEL_DIR, 'train'))
valid_df = load_data(os.path.join(IMAGE_DIR, 'valid'), os.path.join(LABEL_DIR, 'valid'))
test_df = load_data(os.path.join(IMAGE_DIR, 'test'), os.path.join(LABEL_DIR, 'test'))

# Display the first few rows of the DataFrame to verify
print(train_df.head())


                                                path  \
0  dataset/images\train\IMG_2274_jpeg_jpg.rf.2f31...   
1  dataset/images\train\IMG_2275_jpeg_jpg.rf.6635...   
2  dataset/images\train\IMG_2276_jpeg_jpg.rf.7411...   
3  dataset/images\train\IMG_2280_jpeg_jpg.rf.5abc...   
4  dataset/images\train\IMG_2282_jpeg_jpg.rf.510f...   

                                               label  
0  [0, 0.2734375, 0.509765625, 0.33984375, 0.1464...  
1  [0, 0.150390625, 0.75927734375, 0.30078125, 0....  
2  [0, 0.20052083333333334, 0.888671875, 0.230468...  
3  [0, 0.7903645833333334, 0.0498046875, 0.196614...  
4  [3, 0.5989583333333334, 0.482421875, 0.3776041...  


In [6]:
# Function to parse YOLO labels
def parse_yolo_labels(label):
    class_index = label[0]
    bbox_x = label[1]
    bbox_y = label[2]
    bbox_width = label[3]
    bbox_height = label[4]
    
    # You can further process this information as per your model's requirements
    return class_index, bbox_x, bbox_y, bbox_width, bbox_height

In [7]:
# Define data generators (you can use ImageDataGenerator as well)
def data_generator(df, batch_size=32):
    while True:
        df = shuffle(df)
        for start in range(0, len(df), batch_size):
            end = min(start + batch_size, len(df))
            batch_paths = df['path'][start:end]
            batch_labels = df['label'][start:end]
            
            batch_images = []
            batch_parsed_labels = []
            
            for i, path in enumerate(batch_paths):
                image = tf.keras.preprocessing.image.load_img(path, target_size=IMAGE_SIZE)
                image = tf.keras.preprocessing.image.img_to_array(image)
                image = image / 255.0  # Normalize to [0, 1]
                batch_images.append(image)
                
                parsed_label = parse_yolo_labels(batch_labels.iloc[i])
                batch_parsed_labels.append(parsed_label)
            
            yield np.array(batch_images), np.array(batch_parsed_labels)

In [8]:
# Create data iterators
train_generator = data_generator(train_df)
valid_generator = data_generator(valid_df)

In [9]:
# Define the model architecture
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(5)  # Adjust output size as per your label parsing
])

In [10]:
model.compile(optimizer='adam', loss='mse', metrics=['accuracy'])

In [11]:
# Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_df) // 32,
    epochs=10,
    validation_data=valid_generator,
    validation_steps=len(valid_df) // 32
)

Epoch 1/10
13/13 [==============================] - 9s 634ms/step - loss: 187.1658 - accuracy: 0.3942 - val_loss: 7.8367 - val_accuracy: 0.0000e+00
Epoch 2/10
13/13 [==============================] - 7s 528ms/step - loss: 4.9107 - accuracy: 0.1084 - val_loss: 2.5878 - val_accuracy: 0.2396
Epoch 3/10
13/13 [==============================] - 7s 549ms/step - loss: 1.8410 - accuracy: 0.4819 - val_loss: 1.4394 - val_accuracy: 0.5938
Epoch 4/10
13/13 [==============================] - 7s 519ms/step - loss: 1.1496 - accuracy: 0.6530 - val_loss: 1.1404 - val_accuracy: 0.5729
Epoch 5/10
13/13 [==============================] - 7s 522ms/step - loss: 0.9132 - accuracy: 0.6217 - val_loss: 1.0833 - val_accuracy: 0.6146
Epoch 6/10
13/13 [==============================] - 7s 517ms/step - loss: 0.8854 - accuracy: 0.6410 - val_loss: 1.0061 - val_accuracy: 0.5833
Epoch 7/10
13/13 [==============================] - 7s 518ms/step - loss: 0.7770 - accuracy: 0.6627 - val_loss: 0.8835 - val_accuracy: 0.6354


In [12]:
# Evaluate the model
test_images, test_labels = next(data_generator(test_df, len(test_df)))
test_loss, test_acc = model.evaluate(test_images, test_labels)
print(f"Test Loss: {test_loss}, Test Accuracy: {test_acc}")

2/2 [==============================] - 0s 89ms/step - loss: 0.8050 - accuracy: 0.6825
Test Loss: 0.8050200343132019, Test Accuracy: 0.682539701461792


In [13]:
# Make predictions
predictions = model.predict(test_images)

2/2 [==============================] - 0s 90ms/step
